In [67]:
from config_path import add_to_sys_path
add_to_sys_path()  # Call the function to add path

import numpy as np
from sympy.physics.wigner import wigner_3j,wigner_6j
import sympy as sy
from numpy import linalg as LA
from IPython.display import Latex,display
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()
sns.set_palette('terrain')
from Energy_Levels import MoleculeLevels
np.set_printoptions(precision=5, suppress=True)
from Energy_Levels import branching_ratios, Calculate_TDMs

%matplotlib qt

In [68]:
N_list = [0]
print(N_list)

[0]


In [236]:
B = MoleculeLevels.initialize_state('CaOH','40','B000',[0,2],M_values = 'all',I=[0,1/2],S=1/2,round=16)

No P values provided, using P=1/2 as default


In [237]:
B.parameters

{'mu_B': 1.399624494,
 'g_S': 2.0023,
 'g_L': 1,
 '2_e0c': 75346062800.0,
 'mu_N': 0.000762259323,
 'Be': 10230.6874424622,
 'Gamma_SR': -1378.8654313252,
 'bF': 20,
 'c': 50,
 'b': 3.333333333333332,
 'muE': 0.374538528}

In [238]:
# Make a copy for the CaF B state
# Parameters in MHz
CaF_B = B.parameters

# Taken from J. Devlin et al. / Journal of Molecular Spectroscopy 317 (2015) 1–9
CaF_B['Be'] = 0.341259*29979.2458 # = 10230.68744
CaF_B['Gamma_SR'] = -0.045994*29979.2458 # = 1378.86543

# Taken from Tarbutt & Steimle PHYSICAL REVIEW A 92, 053401 (2015)
CaF_B['bF'] = 20
CaF_B['c'] = 50
CaF_B['b'] = CaF_B['bF'] - CaF_B['c']/3

In [239]:
B.update_params(CaF_B,recompute=True)

In [240]:
B.parameters

{'mu_B': 1.399624494,
 'g_S': 2.0023,
 'g_L': 1,
 '2_e0c': 75346062800.0,
 'mu_N': 0.000762259323,
 'Be': 10230.6874424622,
 'Gamma_SR': -1378.8654313252,
 'bF': 20,
 'c': 50,
 'b': 3.333333333333332,
 'muE': 0.374538528}

In [340]:
Bz = np.linspace(1e-3,1000,3000)
Ez = np.linspace(0.0,1000,5000)

In [75]:
help(B.select_q)

Help on method select_q in module Energy_Levels:

select_q(q_dict, evecs=None, parity=None) method of Energy_Levels.MoleculeLevels instance



In [341]:
B.ZeemanMap(Bz,0,plot=True, idx=B.select_q({'N':[0]}))

In [284]:
B.ZeemanMap(Bz,0,plot=True, idx=B.select_q({'N':[2]}))

In [12]:
B.ZeemanMap(Bz,0,plot=True, idx=B.select_q({'N':[1]}))

In [250]:
B.H_symbolic[list(B.select_q({'N':[0],'M':[0]})),:]

Matrix([
[              -15.0, 0, 1.4012340621681*B_z, 0,               0, 0, 0.124846176*E_z, 0, 0, -0.176559155309618*E_z, 0, 0, 0,                      0, 0, 0, 0,                0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[1.4012340621681*B_z, 0,                 5.0, 0, 0.124846176*E_z, 0,               0, 0, 0,                      0, 0, 0, 0, -0.176559155309618*E_z, 0, 0, 0, 11.7851130197758, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [345]:
state = B
thresh = 1e-4
round=5

state.eigensystem(0,1e-3)
M0_idx = state.select_q({'N':[0],'M':[0]})
print(M0_idx)
print('\n')
for i in M0_idx:
    display(Latex('$E = '+str(np.round(state.evals0[i],4))+r'\:\mathrm{MHz}$'))
    print('Decoupled:')
    display(Latex(state.gen_state_str(i,basis='decoupled',thresh=thresh,label_q=['M_N','M_S','M_I','M_F'],round=round)))
    print('Case A:')
    display(Latex(state.gen_state_str(i,basis='aBJ',thresh=thresh,label_q=['Sigma','P','M'],round=round)))
    print('Case B:')
    display(Latex(state.gen_state_str(i,thresh=thresh,round=round)))
    print('\n')

[0 2]




<IPython.core.display.Latex object>

Decoupled:


<IPython.core.display.Latex object>

Case A:


...

<IPython.core.display.Latex object>

Case A:


<IPython.core.display.Latex object>

Case B:


<IPython.core.display.Latex object>

In [346]:
B.evecs_B[2500,:,0]*100

array([ 71.01267,   0.     , -70.40739,  -0.     ,  -0.     ,   0.     ,
        -0.     ,   0.     ,   0.     ,  -0.     ,   0.     ,   0.     ,
         0.     ,  -0.     ,  -0.     ,  -0.     ,   0.     ,   0.     ,
        -0.00007,  -0.     ,   0.     ,   0.     ,  -0.     ,   0.     ,
        -0.00009,   0.     ,   0.     ,   0.     ,  -0.     ,   0.00023,
        -0.     ,   0.     ,   0.     ,  -0.00023,   0.     ,   0.     ])

In [347]:
plt.figure()
plt.plot(Bz,B.evecs_B[:,33,0])
plt.plot(Bz,B.evecs_B[:,29,0])
plt.plot(Bz,B.evecs_B[:,0,0])
plt.plot(Bz,B.evecs_B[:,2,0])
plt.yscale('log')

In [274]:
B.eigensystem(0,1)
B.write_state(17)

E = 60000.873184885735 MHz

 -6.76040257e-08 |K=0,N=0.0,J=0.5,F=1.0,M=-1.0> 

 -2e-16 |K=0,N=0.0,J=0.5,F=1.0,M=1.0> 

 -1e-16 |K=0,N=1.0,J=1.5,F=1.0,M=1.0> 

 4e-16 |K=0,N=1.0,J=1.5,F=2.0,M=-2.0> 

 -1e-16 |K=0,N=1.0,J=1.5,F=2.0,M=0.0> 

 -2e-16 |K=0,N=1.0,J=1.5,F=2.0,M=1.0> 

 -4e-16 |K=0,N=1.0,J=1.5,F=2.0,M=2.0> 

 -0.0003441678728408 |K=0,N=2.0,J=1.5,F=1.0,M=-1.0> 

 -5.5e-15 |K=0,N=2.0,J=1.5,F=1.0,M=1.0> 

 -1e-16 |K=0,N=2.0,J=1.5,F=2.0,M=-2.0> 

 0.0039644643145495 |K=0,N=2.0,J=1.5,F=2.0,M=-1.0> 

 6.49e-14 |K=0,N=2.0,J=1.5,F=2.0,M=1.0> 

 -0.9991445151382404 |K=0,N=2.0,J=2.5,F=2.0,M=-1.0> 

 -1.58375e-11 |K=0,N=2.0,J=2.5,F=2.0,M=1.0> 

 -1e-16 |K=0,N=2.0,J=2.5,F=3.0,M=-2.0> 

 0.0411631199053249 |K=0,N=2.0,J=2.5,F=3.0,M=-1.0> 

 8.156e-13 |K=0,N=2.0,J=2.5,F=3.0,M=1.0> 



In [297]:
X = MoleculeLevels.initialize_state('CaOH','40','X000',[1,3],M_values = 'all',I=[0,1/2],S=1/2,round=8)

No P values provided, using P=1/2 as default


In [298]:
X.parameters

{'mu_B': 1.399624494,
 'g_S': 2.0023,
 'g_L': 1,
 '2_e0c': 75346062800.0,
 'mu_N': 0.000762259323,
 'Be': 10267.5387,
 'D': 0.01154,
 'Gamma_SR': 39.388,
 'bF': 242.91133333333335,
 'c': 401.182,
 'b': 109.184,
 'muE': 0.73749858}

In [299]:
# Make a copy for the CaF X state
# Parameters in MHz
CaF_X = X.parameters

CaF_X['Be'] = 10267.5387
CaF_X['Gamma_SR'] = 39.388

CaF_X['b'] = 109.1840
CaF_X['c'] = 40.1182
CaF_X['bF'] = CaF_X['b'] + CaF_X['c']/3

In [300]:
X.update_params(CaF_X)
X.parameters

{'mu_B': 1.399624494,
 'g_S': 2.0023,
 'g_L': 1,
 '2_e0c': 75346062800.0,
 'mu_N': 0.000762259323,
 'Be': 10267.5387,
 'D': 0.01154,
 'Gamma_SR': 39.388,
 'bF': 122.55673333333333,
 'c': 40.1182,
 'b': 109.184,
 'muE': 0.73749858}

In [351]:
X.ZeemanMap(Bz,0,plot=True, idx=X.select_q({'N':[1],'M':[0]}))

In [ ]:
X.

In [218]:
X.ZeemanMap(Bz,0,plot=True, idx=X.select_q({'N':[1]}))

In [205]:
X.select_q({'N':[1],'M':[0]})

array([ 1,  3,  6, 10])

In [206]:
X.H_symbolic[list(X.select_q({'N':[1],'F':[0,1],'M':[0]})),:]

Matrix([
[0, 0.4670780207227*B_z + 20519.9764066667, 0,                                      0, 146.029335428042 - 0.660548071592424*B_z, 0,                                        0, 0, -1.14410282083972*B_z, 0,                                  0, 0],
[0,                                      0, 0, 20519.9764066667 - 0.4670780207227*B_z,                                        0, 0, 0.660548071592424*B_z + 146.029335428042, 0,                     0, 0,              -1.14410282083972*B_z, 0],
[0,                                      0, 0,                  -1.14410282083972*B_z,                                        0, 0,                    0.404501431495213*B_z, 0,                     0, 0, 0.70061703108405*B_z + 20602.08034, 0]])

In [192]:
X.H_symbolic[list(X.select_q({'N':[1],'M':[0]})),list(X.select_q({'N':[3],'M':[0]}))]

Matrix([
[               0, 0, 0, 0],
[               0, 0, 0, 0],
[               0, 0, 0, 0],
[98.2691193989241, 0, 0, 0]])

In [193]:
list(X.select_q({'N':[3],'M':[0]}))

[35, 41, 48, 56]

In [356]:
X.evecs_B[2900,:,3]*100

array([-82.1433 ,  -0.     ,  -0.     ,  -0.     ,  -0.     ,   0.     ,
        -1.22769,  -0.     ,   0.     ,   0.     , -57.01728,   0.     ,
         0.     ,  -0.     ,   0.     ,   0.     ,   0.     ,   0.     ,
         0.     ,   0.     ,   0.     ,   0.     ,   0.     ,  -0.     ,
        -0.     ,   0.     ,   0.     ,   0.     ,  -0.     ,   0.     ,
         0.     ,  -0.     ,   0.     ,  -0.     ,   0.00005,  -0.     ,
        -0.     ,  -0.     ,  -0.     ,  -0.     ,   0.00007,  -0.     ,
        -0.     ,   0.     ,   0.     ,   0.     ,   0.     ,  -0.     ,
         0.00004,   0.     ,  -0.     ,   0.     ,  -0.     ,  -0.     ,
         0.     ,  -0.     ,   0.00004,  -0.     ,   0.     ,   0.     ])

In [318]:
plt.figure()
plt.plot(Bz,X.evecs_B[:,3,0])
plt.plot(Bz,X.evecs_B[:,3,2])
plt.plot(Bz,X.evecs_B[:,3,5])
plt.plot(Bz,X.evecs_B[:,3,9])
plt.plot(Bz,X.evecs_B[:,3,34],label='N=3')
plt.legend()
plt.yscale('log')

In [194]:
X.write_state(35)

E = 123011.37554761 MHz

 -1e-08 |K=0,N=3.0,J=2.5,F=2.0,M=0.0> 

 0.80373981 |K=0,N=3.0,J=2.5,F=3.0,M=0.0> 

 -0.59498093 |K=0,N=3.0,J=3.5,F=3.0,M=0.0> 



In [113]:
X.H_symbolic[list(X.select_q({'N':[1],'M':[-1,0]})),:]

Matrix([
[0, 0.4670780207227*B_z + 20489.8877566667,                     0,                                      0, 60.9257816210474 - 0.660548071592424*B_z,                   0,                                        0,                                 0, -1.14410282083972*B_z,           0,                     0, 0, -0.173830082344483*E_z,               0,                     0,                0, -0.301082534504527*E_z, 0,                      0, 0,                      0, 0, 0, 0, 0, 0,                      0, 0,                      0, 0, 0, 0,                0, 0,                0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
[0,                                      0,                     0, 20489.8877566667 - 0.4670780207227*B_z,                                        0,                   0, 0.660548071592424*B_z + 60.9257816210474,                                 0,                     0,           0, -1.14410282083972*B_z, 0,                      0,

In [352]:
X.eigensystem(0,500)
M0_idx = X.select_q({'N':[1],'M':[0]})
# M0_idx = list(X.select_q({'N':[3],'M':0}))
print(M0_idx)
print('\n')
for i in M0_idx:
    print(i)
    display(Latex('$E = '+str(np.round(X.evals0[i],4))+r'\:\mathrm{MHz}$'))
    print('Decoupled:')
    display(Latex(X.gen_state_str(i,basis='decoupled',thresh=0.0001,label_q=['K','M_N','M_S','M_I','M_F'],round=4)))
    print('Case A:')
    display(Latex(X.gen_state_str(i,basis='aBJ',thresh=0.0001,label_q=['K','Sigma','P','M'],round=4)))
    print('Case B:')
    display(Latex(X.gen_state_str(i,thresh=0.0001,round=4)))
    print('\n')

[1 3 7 9]


1


<IPython.core.display.Latex object>

Decoupled:


<IPython.core.display.Latex object>

Case A:


...

<IPython.core.display.Latex object>

Case A:


<IPython.core.display.Latex object>

Case B:


<IPython.core.display.Latex object>

In [ ]:
X.write_

In [215]:
X.eigensystem(0,500)
M0_idx = X.select_q({'N':[1],'M':[0]})
print(M0_idx)
print('\n')
for i in M0_idx:
    display(Latex('$E = '+str(np.round(X.evals0[i],4))+r'\:\mathrm{MHz}$'))
    print('Decoupled:')
    display(Latex(X.gen_state_str(i,basis='decoupled',thresh=0.001,label_q=['K','M_N','M_S','M_I','M_F'],round=6)))
    print('Case A:')
    display(Latex(X.gen_state_str(i,basis='aBJ',thresh=0.001,label_q=['K','Sigma','P','M'],round=6)))
    print('Case B:')
    display(Latex(X.gen_state_str(i,thresh=0.001,round=6)))
    print('\n')

[1 3 7 9]




<IPython.core.display.Latex object>

Decoupled:


<IPython.core.display.Latex object>

Case A:


...

<IPython.core.display.Latex object>

Case A:


<IPython.core.display.Latex object>

Case B:


<IPython.core.display.Latex object>

In [197]:
X.eigensystem(0,200)
M0_idx = X.select_q({'N':[3],'M':[0]})
print(M0_idx)
print('\n')
for i in M0_idx:
    display(Latex('$E = '+str(np.round(X.evals0[i],4))+r'\:\mathrm{MHz}$'))
    print('Decoupled:')
    display(Latex(X.gen_state_str(i,basis='decoupled',thresh=0.0001,label_q=['K','M_N','M_S','M_I','M_F'],round=6)))
    print('Case A:')
    display(Latex(X.gen_state_str(i,basis='aBJ',thresh=0.0001,label_q=['K','Sigma','P','M'],round=6)))
    print('Case B:')
    display(Latex(X.gen_state_str(i,thresh=0.0001,round=6)))
    print('\n')

[35 41 49 55]




<IPython.core.display.Latex object>

Decoupled:


<IPython.core.display.Latex object>

Case A:


...

<IPython.core.display.Latex object>

Case A:


<IPython.core.display.Latex object>

Case B:


<IPython.core.display.Latex object>

In [65]:
Bz = np.linspace(1e-6,100000,100001)

In [66]:
X.ZeemanMap(Bz,0,plot=True, idx=X.select_q({'N':[1,3]}))

In [28]:
B000.eigensystem(0,100)
M0_idx = B000.select_q({'N':[0],'M':[-1,0]})
print(M0_idx)
print('\n')
for i in M0_idx:
    display(Latex('$E = '+str(np.round(B000.evals0[i],4))+r'\:\mathrm{MHz}$'))
    print('Decoupled:')
    display(Latex(B000.gen_state_str(i,basis='decoupled',thresh=0.001,label_q=['K','M_N','M_S','M_I','M_F'],round=6)))
    print('Case A:')
    display(Latex(B000.gen_state_str(i,basis='aBJ',thresh=0.001,label_q=['K','Sigma','P','M'],round=6)))
    print('Case B:')
    display(Latex(B000.gen_state_str(i,thresh=0.001,round=6)))
    print('\n')

[0 1 2]




<IPython.core.display.Latex object>

Decoupled:


<IPython.core.display.Latex object>

Case A:


...

<IPython.core.display.Latex object>

Case A:


<IPython.core.display.Latex object>

Case B:


<IPython.core.display.Latex object>

In [32]:
B000.eigensystem(0,900)
M0_idx = B000.select_q({'N':[0],'M':[-1,0]})
print(M0_idx)
print('\n')
for i in M0_idx:
    display(Latex('$E = '+str(np.round(B000.evals0[i],4))+r'\:\mathrm{MHz}$'))
    print('Decoupled:')
    display(Latex(B000.gen_state_str(i,basis='decoupled',thresh=0.001,label_q=['K','M_N','M_S','M_I','M_F'],round=6)))
    print('Case A:')
    display(Latex(B000.gen_state_str(i,basis='aBJ',thresh=0.001,label_q=['K','Sigma','P','M'],round=6)))
    print('Case B:')
    display(Latex(B000.gen_state_str(i,thresh=0.001,round=6)))
    print('\n')

[0 1 2]




<IPython.core.display.Latex object>

Decoupled:


<IPython.core.display.Latex object>

Case A:


...

<IPython.core.display.Latex object>

Case A:


<IPython.core.display.Latex object>

Case B:


<IPython.core.display.Latex object>

In [28]:
X010= MoleculeLevels.initialize_state('CaOH','40','X010',[1],M_values = 'all',I=[0,1/2],S=1/2,round=8)

In [30]:
X010.eigensystem(1000,1e-7)
M0_idx = X010.select_q({'M':[0]})
print(M0_idx)
print('\n')
for i in M0_idx:
    display(Latex('$E = '+str(np.round(X010.evals0[i],4))+r'\:\mathrm{MHz}$'))
    print('Decoupled:')
    display(Latex(X010.gen_state_str(i,basis='decoupled',thresh=0.01,label_q=['K','M_N','M_S','M_I','M_F'],round=4)))
    print('Case A:')
    display(Latex(X010.gen_state_str(i,basis='aBJ',thresh=0.01,label_q=['K','Sigma','P','M'],round=4)))
    print('Case B:')
    display(Latex(X010.gen_state_str(i,thresh=0.01,round=4)))
    print('\n')

[ 2  3  8 11 12 15 18 19]




<IPython.core.display.Latex object>

Decoupled:


<IPython.core.display.Latex object>

Case A:


...

<IPython.core.display.Latex object>

Case A:


<IPython.core.display.Latex object>

Case B:


<IPython.core.display.Latex object>

In [58]:
B000.display_levels(1000,1e-7,'F')

In [10]:
BR1000 = branching_ratios(X010,B000,1000,0)

Successfully converted eigenvectors from bBJ to aBJ
Successfully converted eigenvectors from bBJ to aBJ


In [11]:
def gen_state_str(vector,q_numbers,label_q,thresh=0.02,show_coeff=True):
    full_label = r''
    nonzero_idx = np.nonzero(vector)[0]
    first = 0
    for i,index in enumerate(nonzero_idx):
        coeff = vector[index]
        if abs(coeff) < thresh:
            first+=1
            continue
        sign = {True: '+', False: '-'}[coeff > 0]
        sign0 = {True: ' ', False: sign}[sign=='+']
        if show_coeff:
            label_str = r'${}{}|'.format({True: sign0, False: sign}[i==first],abs(coeff))
        else:
            label_str = r'${}|'.format({True: sign0, False: sign}[i==first])
        val = {q:q_numbers[q][index] for q in label_q}
        for q in label_q:
            _q = {True: '\u039B', False: q}[q=='L']
            _q = {True: '\u03A3', False: _q}[q=='Sigma']
            _q = {True: '\u03A9', False: _q}[q=='Omega']
            if (abs(val[q]) % 1) !=0:
                label_str+=r'{}=\frac{{{}}}{{{}}},'.format(_q,*val[q].as_integer_ratio())
            else:
                label_str+=r'{}={},'.format(_q,int(val[q]))
        label_str = label_str[:-1]+r'\rangle$'
        full_label+=label_str
    if i==first and show_coeff==False:
        full_label = r'$'+full_label[2:]
    return full_label

In [60]:
fig,ax = plt.subplots(figsize=(10,10),constrained_layout=True)
# ax.set_aspect('equal')
log=False
BR_plot = TDM
if log:
    BR_plot=np.log10(BR0)
    vmin = -10
    vmax = 0
else:
    vmin=-1
    vmax = 1
#     log_BR = np.copy(BR0)
#     for i,val1 in enumerate(log_BR):
#         for j,val2 in enumerate(val1):
#             if val2 !=0:
#                 log_BR[i,j] = np.log10(val2)
#     BR_plot = log_BR
mat = ax.matshow(BR_plot,cmap=plt.get_cmap('seismic'),vmax = vmax, vmin = vmin)
y = np.arange(0,len(X010.evecs0))
x = np.arange(0,len(B000.evecs0))
ax.set_yticks(y)
ax.set_xticks(x)
y_labels = [X010.gen_state_str(i,basis='decoupled',thresh=0.05,round=4,label_q=['K','M_N','M_S','M_I','M_F']) for i in range(y.size)]
x_labels = [B000.gen_state_str(i,thresh=0.05,round=4) for i in range(x.size)]
ax.set_yticklabels(y_labels, rotation='40', fontsize=12, ha='right')
ax.set_xticklabels(x_labels, rotation='40', fontsize=12,ha='left')
ax.tick_params(axis='both',labelsize=10,direction='out')
ax.grid(True,which='major',color='white',ls='--',linewidth=0.5)
ax.grid(False,which='minor')
(bot,top) = ax.get_ylim()
ax.set_ylim(bot+0.5, top-0.5)
cbar = fig.colorbar(mat, ax=ax,fraction=0.015, pad=0.05)
#ax.set_xlim(-0.5,x.size+0.5);

C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:32: MatplotlibDeprecationWarning: Auto-removal of grids by pcolor() and pcolormesh() is deprecated since 3.5 and will be removed two minor releases later; please call grid(False) first.


In [59]:
TDM = Calculate_TDMs(0,X010,B000,E0,B0)

Successfully converted eigenvectors from bBJ to aBJ
Successfully converted eigenvectors from bBJ to aBJ
Successfully converted eigenvectors from bBJ to aBJ
Successfully converted eigenvectors from bBJ to aBJ


In [52]:
TDM_0.shape

(24, 16)

In [34]:
fig,ax = plt.subplots(figsize=(10,10),constrained_layout=True)
mat = ax.matshow(TDM_0,cmap=plt.get_cmap('magma'))
cbar = fig.colorbar(mat, ax=ax,fraction=0.015, pad=0.05)
ax.grid(False)

C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:3: MatplotlibDeprecationWarning: Auto-removal of grids by pcolor() and pcolormesh() is deprecated since 3.5 and will be removed two minor releases later; please call grid(False) first.
  This is separate from the ipykernel package so we can avoid doing imports until


In [50]:
TDM_0

array([[ 0.     ,  0.     ,  0.     ,  0.     ],
       [ 0.     ,  0.     , -0.4714 ,  0.     ],
       [ 0.     ,  0.     ,  0.     ,  0.4714 ],
       [ 0.     ,  0.     ,  0.     ,  0.     ],
       [ 0.     ,  0.     ,  0.     ,  0.     ],
       [-0.4714 ,  0.     ,  0.     ,  0.     ],
       [ 0.     ,  0.4714 ,  0.     ,  0.     ],
       [ 0.     ,  0.     ,  0.     ,  0.     ],
       [ 0.57735,  0.     ,  0.     ,  0.     ],
       [ 0.33333,  0.     ,  0.     ,  0.     ],
       [ 0.     ,  0.33333,  0.     ,  0.     ],
       [ 0.     ,  0.57735,  0.     ,  0.     ],
       [ 0.     ,  0.     ,  0.57735,  0.     ],
       [ 0.     ,  0.     ,  0.33333,  0.     ],
       [ 0.     ,  0.     ,  0.     ,  0.33333],
       [ 0.     ,  0.     ,  0.     ,  0.57735]])

In [28]:
X010.select_q({'M':0})

array([ 2,  3,  8, 11, 12, 15, 18, 19])

In [30]:
slice(B000.select_q({'M':0}))

slice(None, array([0, 2]), None)

In [32]:
TDM_0[np.ix_(X010.select_q({'M':1}),B000.select_q({'M':1}))]

array([[0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.]])

In [18]:
B0_idx = B000.select_q({'M':[0]})
X1_idx = 